# A naive Bayes spam filter in NumPy

A naive Bayes filter learns, for each class, how often each word appears, and then combines the
words of a new email with Bayes' theorem as if they were conditionally independent given the
class. Here each email is a row of 0s and 1s, one entry per word in a small vocabulary, and you
build the filter from those rows with NumPy alone: first the fitted probabilities with Laplace
smoothing, then the posterior of each class computed in logarithms.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

words = ["free", "meeting", "winner", "invoice", "lunch", "click"]
# One row per training email: 1 if the word appears in it, 0 if not.
X_train = np.array([
    [1, 0, 1, 0, 0, 1],
    [0, 1, 0, 0, 1, 0],
    [1, 0, 0, 0, 0, 1],
    [0, 1, 0, 1, 0, 0],
    [0, 0, 0, 1, 0, 0],
    [0, 0, 1, 1, 0, 1],
    [1, 0, 0, 0, 1, 0],
    [1, 0, 1, 0, 0, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 1, 0, 1, 0, 1],
])
y_train = np.array([1, 0, 1, 0, 0, 1, 0, 1, 0, 0])   # 1 spam, 0 legitimate
print(X_train.shape, "spam emails:", y_train.sum())

## Part 1 (coding task): Fit the class priors and word probabilities

Write `fit(X, y, alpha=1.0)` for labels 0 and 1. It returns a pair `(log_prior, theta)`:

- `log_prior`, shape `(2,)`: the logarithm of the share of training emails in each class.
- `theta`, shape `(2, d)` for `d` words: `theta[c, j]` estimates the probability that word `j`
  appears in an email of class `c`, with Laplace smoothing,

$$
\theta_{cj} = \frac{(\text{emails of class } c \text{ containing word } j) + \alpha}{(\text{emails of class } c) + 2\alpha} .
$$

With $\alpha = 1$ this is the $(k + 1)/(n + 2)$ of the lesson: one imaginary email with the word
and one without, per class. Row 0 of `theta` is the legitimate class and row 1 is spam.

In [ ]:
def fit(X, y, alpha=1.0):
    """Return (log_prior of shape (2,), theta of shape (2, d))."""
    # TODO
    raise NotImplementedError

In [ ]:
log_prior, theta = fit(X_train, y_train)
assert np.shape(log_prior) == (2,), f"log_prior should have shape (2,), got {np.shape(log_prior)}"
assert np.shape(theta) == (2, 6), f"theta should have shape (2, 6), got {np.shape(theta)}"
assert np.allclose(np.exp(log_prior), [0.6, 0.4]), f"priors {np.exp(log_prior)}: 6 of the 10 emails are legitimate and 4 are spam"
assert np.all(theta > 0) and np.all(theta < 1), "no word may get probability 0 or 1: 'meeting' never appears in the training spam, which is what smoothing is for"
assert not np.allclose(theta[0], [4 / 6, 1 / 6, 4 / 6, 2 / 6, 1 / 6, 4 / 6]), "row 0 is the legitimate class and row 1 is spam"
assert np.allclose(theta[1], [4 / 6, 1 / 6, 4 / 6, 2 / 6, 1 / 6, 4 / 6]), f"spam row {theta[1]}: add alpha to the count and 2 * alpha to the number of spam emails"
assert np.allclose(theta[0], [2 / 8, 5 / 8, 1 / 8, 4 / 8, 3 / 8, 2 / 8]), f"legitimate row {theta[0]}"
_, theta_half = fit(X_train, y_train, alpha=0.5)
assert np.isclose(theta_half[1, 1], 0.5 / 5), "with alpha = 0.5, 'meeting' in spam should be (0 + 0.5) / (4 + 1)"
print("Part 1 passed")

## Part 2 (coding task): Posteriors in logarithms

Write `predict_log_proba(X, log_prior, theta)`. For each row $x$ of `X` it returns
$\log P(c \mid x)$ for $c = 0, 1$, as an array of shape `(m, 2)`. Naive Bayes takes the
likelihood of an email to be a product over all the words, present or absent:

$$
\log P(c) + \sum_j \Big[ x_j \log \theta_{cj} + (1 - x_j) \log (1 - \theta_{cj}) \Big] .
$$

These are the log joint probabilities. Turn them into log posteriors by subtracting, in each
row, the logarithm of the sum of their exponentials. Compute that log-sum-exp stably: subtract
the row's largest value first. Real filters have thousands of words, and the check runs one
with 3,000, where the plain product of probabilities is 0 in both classes.

In [ ]:
def predict_log_proba(X, log_prior, theta):
    """Return log P(class | x) for every row x of X, shape (m, 2)."""
    # TODO
    raise NotImplementedError

In [ ]:
log_prior, theta = fit(X_train, y_train)
email = np.array([[0, 1, 1, 0, 0, 1]])        # meeting, winner, click
post = predict_log_proba(email, log_prior, theta)
assert post.shape == (1, 2), f"one row per email and one column per class: (1, 2), not {post.shape}"
assert np.allclose(np.exp(post).sum(axis=1), 1), "each row should be log probabilities whose exponentials sum to 1"
spam = 0.4 * (2 / 6) * (1 / 6) * (4 / 6) * (4 / 6) * (5 / 6) * (4 / 6)
legit = 0.6 * (6 / 8) * (5 / 8) * (1 / 8) * (4 / 8) * (5 / 8) * (2 / 8)
expected = spam / (spam + legit)
got = np.exp(post[0, 1])
present_only = 0.4 * (1 / 6) * (4 / 6) * (4 / 6) / (0.4 * (1 / 6) * (4 / 6) * (4 / 6) + 0.6 * (5 / 8) * (1 / 8) * (2 / 8))
assert not np.isclose(got, present_only), "the absent words count too: add (1 - x) * log(1 - theta)"
assert np.isclose(got, expected), f"P(spam | email) = {got:.4f}, expected {expected:.4f}"
rng = np.random.default_rng(7)
d, s = 3000, 740
big_theta = np.vstack([np.full(d, 0.2), np.full(d, 0.3)])
x = np.zeros((1, d)); x[0, rng.choice(d, s, replace=False)] = 1
big = predict_log_proba(x, np.log([0.5, 0.5]), big_theta)
assert np.all(np.isfinite(big)), "with 3,000 words the product of probabilities underflows: stay in logs and subtract the row maximum"
log_odds = s * np.log(0.3 / 0.2) + (d - s) * np.log(0.7 / 0.8)
assert np.isclose(np.exp(big[0, 1]), 1 / (1 + np.exp(-log_odds))), "wrong posterior for the 3,000-word email"
batch = predict_log_proba(X_train, log_prior, theta)
assert batch.shape == (10, 2), f"one row per email: (10, 2), not {batch.shape}"
assert np.allclose(np.exp(batch).sum(axis=1), 1), "normalize each email (each row) on its own: the log-sum-exp runs along axis 1"
two = predict_log_proba(np.vstack([x, np.zeros((1, d))]), np.log([0.5, 0.5]), big_theta)
assert np.all(np.isfinite(two)), "subtract each row's own maximum, not one maximum for the whole batch"
print("Part 2 passed")
print(f"P(spam | meeting, winner, click) = {got:.4f}")

## Solution

### Part 1: Fit the class priors and word probabilities

In [ ]:
def fit(X, y, alpha=1.0):
    """Return (log_prior of shape (2,), theta of shape (2, d))."""
    X, y = np.asarray(X, dtype=float), np.asarray(y)
    counts = np.array([(y == c).sum() for c in (0, 1)], dtype=float)        # emails per class
    with_word = np.stack([X[y == c].sum(axis=0) for c in (0, 1)])           # (2, d)
    theta = (with_word + alpha) / (counts[:, None] + 2 * alpha)
    return np.log(counts / counts.sum()), theta

**Why it works.** `X[y == c]` keeps the rows of class `c`, and summing them down the columns counts, for each word,
the emails of that class that contain it. Adding $\alpha$ to that count and $2\alpha$ to the
class size is a Beta$(\alpha, \alpha)$ prior on each word's rate, so a word never seen in the
training spam, like "meeting", gets $1/6$ instead of 0. Without it, a single legitimate word in
a spam email would make the spam likelihood exactly 0, whatever the other words said.

### Part 2: Posteriors in logarithms

In [ ]:
def predict_log_proba(X, log_prior, theta):
    """Return log P(class | x) for every row x of X, shape (m, 2)."""
    X = np.asarray(X, dtype=float)
    joint = log_prior + X @ np.log(theta).T + (1 - X) @ np.log1p(-theta).T    # (m, 2)
    top = joint.max(axis=1, keepdims=True)
    return joint - (top + np.log(np.exp(joint - top).sum(axis=1, keepdims=True)))

**Why it works.** `X @ np.log(theta).T` adds $\log \theta_{cj}$ over the words present in each email, for both classes at
once, and `(1 - X) @ np.log1p(-theta).T` adds $\log(1 - \theta_{cj})$ over the absent ones. Each
row of `joint` then holds the two log joint probabilities, and subtracting their log-sum-exp
divides by $P(x)$, the law of total probability in logs. Subtracting the row maximum first keeps
the largest exponential at exactly 1, so the 3,000-word email, whose probabilities are around
$10^{-737}$, still gives a finite answer. On the small email, "winner" and "click" point to spam
and "meeting" points away from it. The filter gives spam about 0.666, against 0.717 if the
absent words were left out.